# Data Compliance & Regulations for Data Engineers
## GDPR, CCPA, HIPAA, SOX, PCI-DSS & Building Compliant Pipelines

**What you'll learn:**
- Why compliance is YOUR responsibility (not just legal's)
- GDPR: principles, lawful basis, data subject rights, technical requirements
- CCPA/CPRA: California privacy law differences from GDPR
- HIPAA: healthcare data (PHI) handling requirements
- SOX: financial data controls and auditability
- PCI-DSS: payment card data security
- Data classification: public, internal, confidential, restricted
- PII/PHI identification and handling patterns
- Right to deletion (RTBF): implementing in Delta Lake
- Data anonymization & pseudonymization techniques
- Audit trails and lineage for compliance
- Data retention policies and implementation
- Cross-border data transfer rules
- Building compliance INTO pipelines (not bolted on)
- Interview questions

**Estimated Time:** 6-8 hours

---

> Non-compliance can cost MILLIONS in fines (GDPR: up to 4% of global revenue).
> As a DE, you BUILD the systems that either protect or expose data.
> Compliance is not optional -- it's a core engineering requirement.

---

---
# Section 1: The Compliance Landscape

## Why DEs Must Know Compliance

```
WHO IS RESPONSIBLE?

Legal team: Defines policies, interprets regulations
Compliance team: Audits, trains, reports
Data Engineers: IMPLEMENTS the technical controls!
               ↑↑↑ That's YOU! ↑↑↑

Examples of DE compliance work:
  - Implement data deletion when user requests it
  - Encrypt PII columns in the data lake
  - Build audit trails (who accessed what, when)
  - Enforce retention policies (auto-delete after X days)
  - Mask data for non-production environments
  - Implement row-level security (users see only their data)
  - Ensure cross-border transfer rules are followed
```

## Regulation Overview

| Regulation | Scope | Focus | Max Fine |
|-----------|-------|-------|----------|
| **GDPR** | EU/EEA residents' data (worldwide!) | Privacy rights | 4% global revenue or €20M |
| **CCPA/CPRA** | California residents | Consumer privacy | $7,500 per violation |
| **HIPAA** | US healthcare data | Health info protection | $1.5M per violation category |
| **SOX** | US public companies | Financial accuracy | Criminal penalties |
| **PCI-DSS** | Payment card data | Card data security | $100K/month non-compliance |
| **FERPA** | US student education data | Student privacy | Loss of federal funding |
| **COPPA** | US children under 13 | Children's privacy | $50,000 per violation |
| **LGPD** | Brazil residents | Privacy (GDPR-like) | 2% revenue or R$50M |

## Data Classification Framework

```
CLASSIFICATION LEVELS:

PUBLIC:          No restrictions. Marketing content, public APIs.
                 Anyone can access.

INTERNAL:        Company employees only. Meeting notes, org charts.
                 Standard access controls.

CONFIDENTIAL:    Need-to-know basis. Customer data, financial reports.
                 Encrypted, access-logged, role-based access.

RESTRICTED:      Highest sensitivity. PII, PHI, payment cards, secrets.
                 Encrypted at rest + in transit, masked in non-prod,
                 access audited, retention enforced, deletion supported.
```

---
# Section 2: GDPR -- General Data Protection Regulation

## The 7 Principles of GDPR

| # | Principle | What It Means for DE |
|---|-----------|---------------------|
| 1 | **Lawfulness, fairness, transparency** | Must have legal basis to process data |
| 2 | **Purpose limitation** | Collect for specific purpose, don't repurpose without consent |
| 3 | **Data minimization** | Collect ONLY what you need (no "just in case" collection) |
| 4 | **Accuracy** | Keep data correct and up-to-date |
| 5 | **Storage limitation** | Don't keep data longer than needed (retention policies!) |
| 6 | **Integrity & confidentiality** | Protect against unauthorized access (encryption, access control) |
| 7 | **Accountability** | Demonstrate compliance (audit trails, documentation) |

## Data Subject Rights (What Users Can Request)

| Right | What It Means | DE Implementation |
|-------|--------------|-------------------|
| **Right to Access** | User can request all their data | Export pipeline: extract all records for user_id |
| **Right to Rectification** | User can correct their data | Update pipeline: modify specific records |
| **Right to Erasure (RTBF)** | User can request deletion | Delete pipeline: remove from ALL tables/backups |
| **Right to Portability** | User gets data in machine-readable format | Export as JSON/CSV, standard format |
| **Right to Restrict** | User can pause processing | Flag column: is_processing_restricted |
| **Right to Object** | User can opt out of specific processing | Consent management: check before processing |

## Lawful Basis for Processing

| Basis | Example | DE Implication |
|-------|---------|----------------|
| **Consent** | Marketing emails | Must track consent per purpose, honor withdrawal |
| **Contract** | Delivering a purchased product | Can process as needed for service delivery |
| **Legal obligation** | Tax records | Must retain for required period |
| **Legitimate interest** | Fraud prevention | Requires documented impact assessment |
| **Vital interests** | Emergency health situations | Rare, narrow scope |
| **Public task** | Government duties | Usually not DE in private sector |

## Technical Requirements

```
1. ENCRYPTION:
   - At rest: AES-256 for all PII columns/tables
   - In transit: TLS 1.2+ for all data transfers
   - Key management: Rotate keys annually, use KMS

2. ACCESS CONTROL:
   - Role-based access (RBAC) or attribute-based (ABAC)
   - Principle of least privilege
   - Regular access reviews (quarterly)
   - Immediate revocation on role change

3. AUDIT LOGGING:
   - Who accessed what data, when, why
   - Immutable audit trail (cannot be tampered)
   - Retain audit logs for required period (typically 7 years)

4. DATA RETENTION:
   - Automated deletion after retention period
   - Different periods per data type/purpose
   - Backup cleanup (don't forget backups!)

5. BREACH NOTIFICATION:
   - Detect within hours, notify authority within 72 hours
   - Notify affected users "without undue delay"
   - Requires knowing WHAT data was exposed (data catalog!)
```

In [ ]:
# GDPR Right to Erasure implementation pattern
print("IMPLEMENTING RIGHT TO ERASURE (RTBF)")
print("=" * 60)

print("""
CHALLENGE: Delete a user's data from ENTIRE data platform.
           This means: raw, bronze, silver, gold, ML features,
           backups, logs, partner exports, caches...

APPROACH 1: Hard Delete (simple but dangerous)

  -- For each table containing user data:
  DELETE FROM bronze.events WHERE user_id = 'user_123';
  DELETE FROM silver.user_profiles WHERE user_id = 'user_123';
  DELETE FROM gold.customer_metrics WHERE user_id = 'user_123';
  -- Delta Lake: after delete, run VACUUM to physically remove files!
  VACUUM bronze.events RETAIN 0 HOURS;  -- Remove old files immediately

  Problems:
    - Must know ALL tables containing user data (data catalog required!)
    - Delta Lake keeps history (time travel) -- must VACUUM
    - Backups must also be cleaned
    - Downstream systems (partner exports, caches) must be notified

APPROACH 2: Crypto-Shredding (recommended for large scale)

  -- Instead of deleting records, encrypt PII with a per-user key
  -- To "delete", destroy the encryption key → PII is unreadable

  Table: users
    user_id | encrypted_name | encrypted_email | encryption_key_id
    123     | aX7f...       | bY8g...        | key_123

  Key Store:
    key_123 → AES-256 key for user 123

  DELETION = delete key_123 from key store
  → encrypted columns become permanently unreadable
  → No need to find/modify every table!
  → Analytics on non-PII columns still work

APPROACH 3: Pseudonymization + Deletion Request Queue

  1. Replace PII with pseudonymous IDs at ingestion (Bronze→Silver)
     user_email → hashed_id (irreversible without mapping table)
  2. Keep mapping table separately (highly restricted)
  3. Deletion = delete from mapping table + anonymize remaining

  Benefit: Analytics still work on pseudonymous data
  Delta Lake: gold layer never had real PII to begin with
""")

print("IMPLEMENTATION CHECKLIST:")
checklist = [
    "Maintain a DATA MAP: which tables contain PII, which columns",
    "Automate: RTBF request → scan all tables → delete → confirm",
    "Track: request_received_date, completed_date (must be within 30 days)",
    "Verify: after deletion, query all tables to confirm removal",
    "Document: audit log of what was deleted, when, by whom",
    "Backups: ensure backup retention aligns with RTBF capability",
    "Third parties: notify data processors to also delete",
    "Delta Lake: VACUUM after delete to remove physical files",
]
for i, item in enumerate(checklist, 1):
    print(f"  {i}. {item}")

---
# Section 3: HIPAA & PCI-DSS

## HIPAA (Health Insurance Portability and Accountability Act)

### What is PHI (Protected Health Information)?

PHI = health data that can identify an individual. Includes:

| PHI Element | Example | DE Handling |
|-------------|---------|-------------|
| Name | John Smith | Encrypt or de-identify |
| DOB | 1990-05-15 | Generalize to year or age band |
| SSN | 123-45-6789 | Never store plain text! |
| Medical Record # | MRN-12345 | Pseudonymize |
| Health plan ID | BCBS-9876 | Encrypt |
| Diagnosis codes | ICD-10: E11.9 | Can aggregate (not individually identifiable alone) |
| Treatment records | "Prescribed X" | Encrypt, restrict access |
| Dates (admission, discharge) | 2024-01-15 | Generalize for research use |

### HIPAA Safe Harbor De-identification

Remove these 18 identifiers to make data "de-identified" (safe to share):
```
1. Names                    10. Account numbers
2. Addresses (below state)  11. Certificate/license numbers
3. Dates (except year)      12. Vehicle identifiers
4. Phone numbers            13. Device identifiers
5. Fax numbers              14. Web URLs
6. Email addresses          15. IP addresses
7. SSN                      16. Biometric identifiers
8. Medical record numbers   17. Full-face photos
9. Health plan numbers      18. Any other unique identifier
```

### HIPAA Technical Safeguards

| Requirement | Implementation |
|------------|----------------|
| Access control | RBAC, unique user IDs, auto-logoff |
| Audit controls | Log all PHI access (who, what, when) |
| Integrity | Ensure data not altered improperly (checksums) |
| Transmission security | Encrypt all PHI in transit (TLS 1.2+) |
| Encryption at rest | AES-256 for stored PHI |
| BAA (Business Associate Agreement) | Required for any vendor handling PHI |

## PCI-DSS (Payment Card Industry Data Security Standard)

### What Must Be Protected

| Data Element | Storage Allowed? | Must Encrypt? |
|-------------|-----------------|---------------|
| Primary Account Number (PAN) | Yes (if encrypted) | YES (AES-256) |
| Cardholder Name | Yes | Recommended |
| Expiration Date | Yes | Recommended |
| CVV/CVC | NEVER store after authorization! | N/A (don't store!) |
| PIN | NEVER store! | N/A (don't store!) |
| Full magnetic stripe | NEVER store! | N/A (don't store!) |

### PCI-DSS for DEs: Key Rules

```
1. NEVER store CVV, PIN, or full track data (even encrypted)
2. Mask PAN in displays: show only last 4 digits (****-****-****-1234)
3. Encrypt PAN at rest with AES-256
4. Tokenize: replace PAN with token for analytics (token has no value if stolen)
5. Limit access: only systems that NEED the PAN can access it
6. Network segmentation: PCI systems in isolated network segment
7. Logging: log all access to cardholder data
8. Retention: delete card data as soon as business need expires
```

In [ ]:
# Data anonymization techniques
print("DATA ANONYMIZATION & PSEUDONYMIZATION TECHNIQUES")
print("=" * 60)

import hashlib

print("""
TECHNIQUE COMPARISON:

| Technique | Reversible? | Utility Preserved? | Use Case |
|-----------|------------|-------------------|----------|
| Encryption | Yes (with key) | No (ciphertext) | Storage protection |
| Hashing | No (one-way) | Low | Pseudonymization, dedup |
| Tokenization | Yes (with vault) | No | Payment cards |
| Masking | No | Partial | Display, non-prod |
| Generalization | No | Medium | Analytics, research |
| K-anonymity | No | Medium | Publishing datasets |
| Differential privacy | No | High | Statistics, ML |
""")

# Demonstrate techniques
print("EXAMPLES:")

# 1. Hashing (pseudonymization)
email = "john.doe@example.com"
hashed = hashlib.sha256(email.encode()).hexdigest()[:16]
print(f"\n  1. HASHING (pseudonymization):")
print(f"     Original: {email}")
print(f"     Hashed:   {hashed}")
print(f"     Use: join across tables without exposing email")

# 2. Masking
phone = "+1-555-123-4567"
masked = phone[:7] + "***-" + phone[-4:]
print(f"\n  2. MASKING:")
print(f"     Original: {phone}")
print(f"     Masked:   {masked}")

# 3. Generalization
age = 34
age_band = f"{(age // 10) * 10}-{(age // 10) * 10 + 9}"
print(f"\n  3. GENERALIZATION:")
print(f"     Original age: {age}")
print(f"     Generalized:  {age_band}")
print(f"     Zip 10001 -> State: NY")

# 4. Tokenization
card = "4532-1234-5678-9012"
import uuid
token = str(uuid.uuid4())[:12]
print(f"\n  4. TOKENIZATION:")
print(f"     Card PAN:  {card}")
print(f"     Token:     tok_{token}")
print(f"     (Vault maps token back to PAN, only payment service has access)")

# 5. K-anonymity
print(f"\n  5. K-ANONYMITY (k=3):")
print(f"     Before: Age 34, Zip 10001, Gender M -> unique!")
print(f"     After:  Age 30-39, Zip 100**, Gender M -> at least 3 people match")
print(f"     Guarantee: any record is indistinguishable from k-1 others")

---
# Section 4: SOX Compliance & Audit Trails

## SOX (Sarbanes-Oxley) for DEs

SOX applies to financial reporting at public companies. DEs must ensure:

| Requirement | DE Implementation |
|------------|-------------------|
| **Accuracy** | Data quality checks on financial data, reconciliation |
| **Completeness** | No dropped records in financial pipelines |
| **Auditability** | Full lineage: source → transform → report |
| **Change control** | CI/CD with approval gates for financial pipelines |
| **Access control** | Separation of duties (dev can't deploy to prod alone) |
| **Retention** | Financial records retained 7 years minimum |
| **Immutability** | Financial source data cannot be modified after close |

## Building Audit Trails

```sql
-- Audit trail table pattern (append-only!)
CREATE TABLE audit.data_access_log (
    event_id STRING,           -- UUID
    event_timestamp TIMESTAMP, -- When
    user_id STRING,            -- Who
    action STRING,             -- What (READ, WRITE, DELETE, EXPORT)
    resource_type STRING,      -- Table, file, API
    resource_name STRING,      -- e.g., "gold.customer_pii"
    row_count INT,             -- How many records affected
    query_text STRING,         -- The actual query (for WRITE/DELETE)
    ip_address STRING,         -- From where
    justification STRING,      -- Why (required for PII access)
    data_classification STRING -- confidential, restricted, etc.
);

-- CRITICAL: This table must be:
-- 1. APPEND-ONLY (no updates or deletes!)
-- 2. In a separate schema with restricted access
-- 3. Retained for 7+ years (SOX) or per regulation
-- 4. Backed up independently
-- 5. Tamper-evident (checksums or blockchain-like chain)
```

## Data Lineage for Compliance

```
SOURCE (what data?)     TRANSFORM (what changed?)     DESTINATION (where?)
  ┌───────────┐          ┌───────────────┐           ┌───────────┐
  │ raw.orders│─────────>│ Spark job:    │─────────>│ gold.     │
  │           │          │ deduplicate,  │           │ revenue   │
  │ Contains: │          │ filter nulls, │           │           │
  │ - PII ✓   │          │ aggregate by  │           │ Contains: │
  │ - PCI ✗   │          │ customer      │           │ - PII ✓   │
  │           │          │               │           │ (custID)  │
  └───────────┘          └───────────────┘           └───────────┘
       │                        │                          │
       └────────────────────────┼──────────────────────────┘
                                │
                    LINEAGE METADATA (Unity Catalog):
                    "gold.revenue derives from raw.orders
                     via job 'daily_revenue_etl' (v2.3.1)
                     last run: 2024-06-15 02:00 UTC
                     row count: 5,000 in, 4,850 out (150 filtered)"
```

In [ ]:
# Data retention implementation
print("DATA RETENTION POLICY IMPLEMENTATION")
print("=" * 60)

print("""
RETENTION POLICY TABLE:

| Data Category | Retention Period | Action After | Regulation |
|--------------|-----------------|--------------|------------|
| Transaction records | 7 years | Archive then delete | SOX |
| Customer PII | Duration of relationship + 30 days | Delete | GDPR |
| Health records (PHI) | 6 years from last service | Delete | HIPAA |
| Payment card data | Until transaction settled | Delete immediately | PCI-DSS |
| Marketing consent | Until withdrawn + 30 days | Delete | GDPR |
| Audit logs | 7 years | Archive (never delete) | SOX/GDPR |
| Employee data | Employment + 7 years | Archive then delete | Labor law |
| Anonymized analytics | Indefinite | No action needed | N/A (not personal) |
| Raw event logs | 90 days | Delete or anonymize | Internal policy |
| ML training data | Until model retrained | Re-anonymize or delete PII | GDPR Art. 17 |

IMPLEMENTATION PATTERN (Delta Lake):

  -- 1. Tag tables with retention metadata
  ALTER TABLE bronze.events SET TBLPROPERTIES (
      'retention_days' = '90',
      'data_classification' = 'confidential',
      'contains_pii' = 'true',
      'deletion_policy' = 'hard_delete'
  );

  -- 2. Automated retention enforcement job (runs daily):
  -- For each table with retention policy:
  DELETE FROM bronze.events
  WHERE event_date < CURRENT_DATE - INTERVAL retention_days DAYS;
  VACUUM bronze.events RETAIN 0 HOURS;

  -- 3. Log the deletion in audit trail
  INSERT INTO audit.retention_log VALUES (
      uuid(), current_timestamp(), 'retention_job',
      'bronze.events', rows_deleted, '90-day retention policy'
  );
""")

# Cross-border transfer rules
print("\nCROSS-BORDER DATA TRANSFER:")
print("""
  GDPR restricts transferring EU data outside EU/EEA unless:
  1. Adequacy decision (destination country deemed safe: UK, Japan, etc.)
  2. Standard Contractual Clauses (SCCs) with data importer
  3. Binding Corporate Rules (for intra-company transfers)
  4. Explicit consent (limited, case-by-case)

  DE IMPLICATIONS:
  - AWS region matters! EU data → eu-west-1 (Ireland), NOT us-east-1
  - Replication across regions? Must have legal basis
  - Data residency requirements: some data MUST stay in-country
  - Cloud provider must have Data Processing Agreement (DPA)
  - Log all cross-border transfers in audit trail
""")

---
# Section 6: Building Compliance INTO Pipelines

## Compliance-by-Design Architecture

```
INGESTION (Bronze):
  ┌─────────────────────────────────────────┐
  │ 1. CLASSIFY at ingestion:               │
  │    - Tag PII/PHI columns automatically  │
  │    - Apply data classification label    │
  │ 2. ENCRYPT PII columns (column-level)   │
  │ 3. LOG: source, timestamp, schema       │
  └─────────────────────────────────────────┘
           │
PROCESSING (Silver):
  ┌─────────────────────────────────────────┐
  │ 4. PSEUDONYMIZE: replace PII with IDs   │
  │ 5. CONSENT CHECK: only process if legal │
  │    basis exists for this purpose         │
  │ 6. QUALITY: validate PII formats        │
  │ 7. LINEAGE: track transformations       │
  └─────────────────────────────────────────┘
           │
SERVING (Gold):
  ┌─────────────────────────────────────────┐
  │ 8. ACCESS CONTROL: RBAC/ABAC per column │
  │ 9. MASKING: non-privileged users see    │
  │    masked PII (Unity Catalog policies)  │
  │ 10. AUDIT: log all queries on PII tables│
  └─────────────────────────────────────────┘
           │
DELETION:
  ┌─────────────────────────────────────────┐
  │ 11. RTBF: automated deletion pipeline   │
  │ 12. VACUUM: physical file cleanup       │
  │ 13. VERIFY: confirm deletion complete   │
  │ 14. NOTIFY: confirm to data subject     │
  └─────────────────────────────────────────┘
```

## Unity Catalog + Column-Level Security (Databricks)

```sql
-- Column masking: non-privileged users see masked data
CREATE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN CASE
    WHEN is_member('pii_admins') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@***.com')
END;

ALTER TABLE gold.customers
ALTER COLUMN email SET MASK mask_email;

-- Row-level security: users see only their region's data
CREATE FUNCTION region_filter(region STRING)
RETURNS BOOLEAN
RETURN (is_member('global_admins') OR region = current_user_region());

ALTER TABLE gold.customers SET ROW FILTER region_filter ON (region);
```

In [ ]:
print("COMPLIANCE INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "How would you implement Right to Erasure (GDPR Article 17) in a data lake?",
        "a": "1) Maintain a data catalog mapping user_id → all tables/columns containing their data. 2) On deletion request, execute DELETE on each table for that user_id. 3) Run VACUUM on Delta tables to physically remove old files. 4) Handle backups (either exclude user from future restores or apply deletion to backup copies). 5) Notify downstream systems and third-party processors. 6) Log the deletion in audit trail. 7) Confirm completion within 30 days. Alternative: crypto-shredding (destroy per-user encryption key)."
    },
    {
        "q": "How do you handle PII in non-production environments?",
        "a": "NEVER copy production PII to dev/staging. Options: 1) Synthetic data generation (Faker library, realistic but fake). 2) Data masking pipeline (replace PII before copying: names→random, emails→hashed, dates→shifted). 3) Subset + anonymize (take 1% sample, strip all PII columns). 4) Row-level access in Unity Catalog that blocks dev roles from PII. The key: non-prod environments should never contain real identifiable data."
    },
    {
        "q": "What's the difference between anonymization and pseudonymization?",
        "a": "Anonymization: IRREVERSIBLE. Cannot identify individual even with additional info. Data is no longer 'personal data' under GDPR (no restrictions apply). Example: aggregate statistics, k-anonymity. Pseudonymization: REVERSIBLE with additional info (the key/mapping). Still personal data under GDPR but qualifies for lighter requirements. Example: replace email with user_id (mapping table exists separately)."
    },
    {
        "q": "How do you ensure audit compliance for data access?",
        "a": "1) Unity Catalog audit logs (automatic for Databricks). 2) Custom audit table for application-level access (append-only, immutable). 3) Log: who, what table, what query, when, from where, how many rows. 4) Store audit logs in a separate, restricted schema (auditors only). 5) Retain for 7+ years (SOX requirement). 6) Regular review: generate reports of PII access for compliance team. 7) Alert on anomalous access patterns (bulk PII export by unusual user)."
    },
    {
        "q": "How would you design a data retention automation system?",
        "a": "1) Table metadata: tag each table with retention_days, data_class, deletion_policy. 2) Daily job scans all tables for records past retention date. 3) Soft-delete first (mark as expired), hard-delete after grace period. 4) Run VACUUM to physically remove. 5) Log all deletions in audit trail. 6) Alert if retention job fails (compliance risk). 7) Exclude from deletion: data under legal hold, active RTBF requests. 8) Report: monthly compliance status to legal team."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: Compliance Cheat Sheet for DEs

## What You MUST Implement

| Requirement | Implementation | Tools |
|-------------|---------------|-------|
| Encryption at rest | Column-level or table-level AES-256 | KMS, Delta Lake encryption |
| Encryption in transit | TLS 1.2+ for all connections | VPC endpoints, SSL configs |
| Access control | RBAC/ABAC, least privilege | Unity Catalog, Lake Formation, IAM |
| Audit logging | Immutable access logs | Unity Catalog audit, custom tables |
| Data masking | Dynamic column masking for non-privileged | Unity Catalog masking functions |
| Retention enforcement | Automated deletion after policy period | Scheduled Spark jobs + VACUUM |
| Right to deletion | Automated RTBF pipeline | Delta DELETE + VACUUM + verification |
| Data classification | Tag all tables/columns with sensitivity | Data catalog, table properties |
| Cross-border controls | Region-specific storage | Multi-region architecture |
| Breach detection | Anomaly alerting on data access | CloudTrail + custom monitoring |

## Quick Reference: "Can I Store This?"

| Data Type | Store? | Requirements |
|-----------|--------|-------------|
| Email address | Yes | Encrypt, access control, deletable |
| Credit card number | Only if PCI-compliant | Tokenize preferred, encrypt if stored |
| CVV/PIN | NEVER | Do not store under any circumstance |
| SSN | Minimize | Encrypt, strict access, consider not storing |
| Health diagnosis | Yes (with HIPAA controls) | Encrypt, BAA required, audit access |
| Age/gender | Yes | Low sensitivity, generalize for sharing |
| Browsing history | Yes (with consent) | Purpose-limited, retention policy |
| Biometric data | Extreme caution | Special category under GDPR |

---
*Data Compliance & Regulations for Data Engineers*